# Nelson-Siegel Factor Calibration & Out-of-Sample Yield Curve Forecasting

End-to-end research pipeline: daily Nelson-Siegel factor estimation across the `tau1` grid, expanding-window walk-forward forecasting (`AR(1)`, `VAR`, `VECM`, `Random Forest`, `CatBoost`), yield curve reconstruction across maturities (`0.25Y - 10Y`), and `RMSE` / Diebold-Mariano evaluation against Random Walk.

In [ ]:
# Импорты и настройки
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from datetime import datetime
import statsmodels.api as sm
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

# Импорт наших функций
from ofz_curve import *
from ofz_curve.paths import DYNAMIC_TAU_PARAMS_CSV, WRITING_LATEX_GENERATED_DIR

import catboost

# --- Подавление Предупреждений ---
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print("Библиотеки и функции импортированы, настройки загружены.")


In [ ]:
# === 1. Загрузка и подготовка данных ===
df_filtered = load_ofz_data()
oil_series_full = load_oil_data()
usd_series_full = load_usd_data()


In [ ]:
# === Выбор сетки tau1 ===
# Для воспроизводимости текущего текста исследования фиксируем расширенную сетку, как в приложении:
TAU1_VALUES_TO_TEST = [1.0, 1.4, 1.8, 2.2, 2.6, 3.0, 3.2, 3.4, 3.6, 3.8, 4.0, 5.0]

# Альтернатива для обновлённой версии исследования (потребует пересборки таблиц/текста исследования):
# TAU1_VALUES_TO_TEST = optimize_tau1(df_filtered)

print(f"Будем тестировать tau1: {TAU1_VALUES_TO_TEST}")


In [ ]:
# === 2. Анализ для различных значений tau1 ===
all_tau1_results = {}
all_tau1_rmse_results = {}

print(f"Будет проанализировано значений tau1: {len(TAU1_VALUES_TO_TEST)}")
print(f"Горизонты прогнозирования: {FORECAST_HORIZONS_DAYS} дней")

# Цикл по различным значениям tau1
for FIXED_TAU1_NS in TAU1_VALUES_TO_TEST:
    print(f"\n{'='*70}")
    print(f"НАЧАЛО АНАЛИЗА ДЛЯ TAU1 = {FIXED_TAU1_NS}")
    print(f"{'='*70}")
    
    # Калибровка NS и прогнозирование
    params_successful, ar_factor_forecasts_df, yield_forecasts_df, rmse_df = calibrate_ns_for_tau1(
        df_filtered, FIXED_TAU1_NS
    )
    
    # Сохраняем результаты
    all_tau1_rmse_results[FIXED_TAU1_NS] = rmse_df
    all_tau1_results[FIXED_TAU1_NS] = {
        'params_successful': params_successful,
        'ar_factor_forecasts_df': ar_factor_forecasts_df,
        'yield_forecasts_df': yield_forecasts_df,
        'rmse_df': rmse_df
    }

print(f"\n{'='*70}")
print("АНАЛИЗ ДЛЯ ВСЕХ ЗНАЧЕНИЙ TAU1 ЗАВЕРШЕН")
print(f"{'='*70}")


In [ ]:
# === Сводка и визуализация RMSE по tau1 ===
if all_tau1_rmse_results:
    combined_rmse_df, summary_df, best_tau1, best_avg_rmse = analyze_tau1_results(all_tau1_rmse_results)
    
    # Визуализация
    plot_rmse_comparison(combined_rmse_df, summary_df, best_tau1, best_avg_rmse, save_path=ROOT / 'writing' / 'latex' / 'media' / 'media' / 'image3.png')
    
    print("\nАНАЛИЗ ЗАВЕРШЕН")
else:
    print("Нет результатов для анализа RMSE")


In [ ]:
# === Прогнозирование факторов различными моделями ===
# Используем лучшее tau1 для дальнейшего анализа
best_tau1_data = all_tau1_results.get(best_tau1, {})
if 'params_successful' in best_tau1_data:
    factors_df = best_tau1_data['params_successful']
    
    # Подготовка данных для VECM
    factors_for_vecm_df = prepare_factors_for_vecm(factors_df, oil_series_full, usd_series_full)
    
    # AR(1) прогнозирование
    ar_factor_forecasts_df = forecast_factors_ar1(factors_df)
    
    # VAR в уровнях
    var_factor_forecasts_df = forecast_factors_var_level(factors_df, oil_series_full, usd_series_full)
    
    # VAR_diff_level
    var_dl_factor_forecasts_df = forecast_factors_var_dl(factors_df, oil_series_full, usd_series_full)
    
    # VECM
    vecm_factor_forecasts_df, vecm_fit_success, vecm_fit_failures, johansen_msg, trace_rank, max_eig_rank = forecast_factors_vecm(factors_for_vecm_df)
    
    # ML модель - CatBoost (полное покрытие OOS)
    import importlib
    import ofz_curve.ml_models as _ml
    importlib.reload(_ml)
    catboost_factor_forecasts_df, catboost_models = _ml.forecast_factors_catboost(
        factors_df,
        oil_series_full,
        usd_series_full,
        oos_step=1,
        oos_max_points=None,  # без усечения хвоста
        retrain_every=20,
        cb_iterations=250,
        cb_learning_rate=0.06
    )
    
    print(f"\nРезультаты прогнозирования:")
    print(f"AR(1): {len(ar_factor_forecasts_df)} прогнозов")
    print(f"VAR_level: {len(var_factor_forecasts_df)} прогнозов")
    print(f"VAR_dl: {len(var_dl_factor_forecasts_df)} прогнозов")
    print(f"VECM: {len(vecm_factor_forecasts_df)} прогнозов")
    print(f"CatBoost: {len(catboost_factor_forecasts_df)} прогнозов")
else:
    print("Нет данных факторов для прогнозирования")


In [ ]:
# === IRF: импульсные отклики ===
irf_yield_df, var_res_irf = analyze_irf(factors_df, oil_series_full, usd_series_full)

# Визуализация IRF
plot_irf_yields(irf_yield_df, [126, 252])

print("IRF анализ завершен")


In [ ]:
# === Сравнение прогнозов доходностей по всем моделям ===
# Объединяем прогнозы факторов
merged_fc_df = merge_factor_forecasts(
    ar_factor_forecasts_df,
    var_factor_forecasts_df,
    var_dl_factor_forecasts_df,
    vecm_factor_forecasts_df,
    catboost_factor_forecasts_df
)

print(f"Объединено {len(merged_fc_df)} прогнозов факторов")

# Сравнение ML vs эконометрических моделей: все факторы и горизонты
if not catboost_factor_forecasts_df.empty and not ar_factor_forecasts_df.empty:
    comparison_all = _ml.compare_ml_vs_econometric_all(
        ar_factor_forecasts_df,
        catboost_factor_forecasts_df,
        factors=['beta0', 'beta1', 'beta2'],
        horizons=FORECAST_HORIZONS_DAYS,
    )
    if not comparison_all.empty:
        rmse_summary = comparison_all.pivot_table(
            values='RMSE',
            index=['factor', 'horizon_days'],
            columns='Model'
        ).reset_index()
        print()
        print("Сравнение CatBoost vs AR(1) по всем факторам/горизонтам (RMSE):")
        print(rmse_summary.to_string(index=False))

        avg_rmse = rmse_summary[['AR(1)', 'CatBoost']].mean().to_dict()
        print(f"Средний RMSE AR(1): {avg_rmse.get('AR(1)', float('nan')):.4f}")
        print(f"Средний RMSE CatBoost: {avg_rmse.get('CatBoost', float('nan')):.4f}")
    else:
        print("Нет пересечений дат/факторов для сравнения CatBoost vs AR(1)")
else:
    print("Нет данных для сравнения CatBoost vs AR(1)")

# Расчет прогнозов доходностей и RMSE по моделям
yield_forecasts_full_df, model_forecast_col_map = calculate_yield_forecasts(merged_fc_df, best_tau1, df_filtered)
rmse_summary_df = calculate_rmse_summary(yield_forecasts_full_df)
plot_yield_forecast_rmse(rmse_summary_df, EVAL_MATURITIES_YEARS, FORECAST_HORIZONS_DAYS)


In [ ]:
# === Робастность: динамический tau1 (проверка устойчивости) ===
import importlib
import ofz_curve.dyn_tau_analysis as _dyn
importlib.reload(_dyn)

dyn_params_df = _dyn.load_dyn_params_df(str(DYNAMIC_TAU_PARAMS_CSV))
dyn_eda_stats = _dyn.run_dyn_tau_eda(dyn_params_df)

dyn_factor_forecasts_df, yield_forecasts_dyn_df, rmse_dyn_yield_df = _dyn.forecast_dynamic_tau1(
    dyn_params_df=dyn_params_df,
    df_filtered=df_filtered,
    horizons_days=FORECAST_HORIZONS_DAYS,
    out_of_sample_ratio=OUT_OF_SAMPLE_START_RATIO,
    rolling_window=5,
    eval_maturities_years=EVAL_MATURITIES_YEARS,
)

if rmse_dyn_yield_df.empty or rmse_summary_df.empty:
    print("Нет данных для сравнения динамического tau1 со статической спецификацией")
else:
    static_rmse = rmse_summary_df[[
        'Horizon (days)',
        'Maturity (years)',
        'RMSE_AR1',
    ]].rename(columns={
        'Horizon (days)': 'horizon_days',
        'Maturity (years)': 'maturity_years',
    })

    dyn_vs_static_rmse_df = pd.merge(
        static_rmse,
        rmse_dyn_yield_df[['horizon_days', 'maturity_years', 'RMSE_NS_DYNAMIC']],
        on=['horizon_days', 'maturity_years'],
        how='inner',
    )

    dyn_vs_static_rmse_df['improvement_pct'] = (
        (dyn_vs_static_rmse_df['RMSE_AR1'] - dyn_vs_static_rmse_df['RMSE_NS_DYNAMIC'])
        / dyn_vs_static_rmse_df['RMSE_AR1']
        * 100
    )

    mean_static = dyn_vs_static_rmse_df['RMSE_AR1'].mean()
    mean_dyn = dyn_vs_static_rmse_df['RMSE_NS_DYNAMIC'].mean()
    better_cells = int((dyn_vs_static_rmse_df['RMSE_NS_DYNAMIC'] < dyn_vs_static_rmse_df['RMSE_AR1']).sum())
    total_cells = int(len(dyn_vs_static_rmse_df))

    print()
    print(f"Итог сравнения dynamic tau1 vs static tau1={best_tau1} (базовая NS-AR1):")
    print(f"Средняя RMSE static: {mean_static:.4f}")
    print(f"Средняя RMSE dynamic: {mean_dyn:.4f}")
    print(f"Среднее улучшение (%): {dyn_vs_static_rmse_df['improvement_pct'].mean():.4f}")
    print(f"Dynamic лучше в {better_cells}/{total_cells} ячейках")

    horizon_cmp = (
        dyn_vs_static_rmse_df
        .groupby('horizon_days')[['RMSE_AR1', 'RMSE_NS_DYNAMIC']]
        .mean()
        .reset_index()
    )
    horizon_cmp['improvement_pct'] = (
        (horizon_cmp['RMSE_AR1'] - horizon_cmp['RMSE_NS_DYNAMIC'])
        / horizon_cmp['RMSE_AR1']
        * 100
    )

    display(dyn_vs_static_rmse_df.sort_values(['horizon_days', 'maturity_years']))
    display(horizon_cmp)

    WRITING_LATEX_GENERATED_DIR.mkdir(parents=True, exist_ok=True)
    out_dyn_vs_static = WRITING_LATEX_GENERATED_DIR / 'dyn_vs_static_rmse.csv'
    out_dyn_rmse = WRITING_LATEX_GENERATED_DIR / 'dyn_tau_rmse_table.csv'
    dyn_vs_static_rmse_df.to_csv(out_dyn_vs_static, index=False)
    rmse_dyn_yield_df.to_csv(out_dyn_rmse, index=False)
    print(f"CSV сохранены: {out_dyn_vs_static}, {out_dyn_rmse}")


In [ ]:
# === Визуализация фактора: факт vs все модели + ошибки ===
# Обновляем функции на случай правок в коде
import importlib
import ofz_curve.visualization as _viz
importlib.reload(_viz)
from ofz_curve.visualization import plot_factor_forecasts_combined

# По умолчанию берём beta0 и горизонт 21д; берём хвост 250 точек для читаемости.
plot_factor_forecasts_combined(
    merged_fc_df,
    factor='beta0',
    horizon_days=21,
    last_n_points=250,
    thin_every=2,
)


In [ ]:
# === Визуализация прогнозов доходности (фиксированный срок/горизонт) ===
from nelson_siegel_svensson import NelsonSiegelCurve


def _interp_actual_yield(df_filtered, target_date, maturity_years):
    """Интерполяция фактической доходности для даты и срока погашения."""
    day_data = df_filtered[df_filtered['TRADEDATE'] == target_date]
    if day_data.empty:
        return np.nan

    ttms = day_data['TTM_years'].values
    yields = day_data['YIELDCLOSE'].values
    sort_idx = np.argsort(ttms)
    ttms, yields = ttms[sort_idx], yields[sort_idx]

    if len(ttms) >= 2:
        return float(np.interp(maturity_years, ttms, yields, left=yields[0], right=yields[-1]))
    if len(ttms) == 1 and np.isclose(ttms[0], maturity_years):
        return float(yields[0])
    return np.nan


def plot_yield_forecasts_timeseries(merged_fc_df, df_filtered, tau1, maturity_years=3.0,
                                    horizon_days=21, last_n_points=250):
    """Визуализация фактических и прогнозных доходностей по выбранному сроку/горизонту."""
    if merged_fc_df.empty:
        print("merged_fc_df пуст - нет прогнозов факторов")
        return

    model_cols = {
        'AR(1)': 'forecast_value_ar1',
        'VAR_level': 'forecast_value_var_level',
        'VAR_dl': 'forecast_value_var_dl',
        'VECM': 'forecast_value_vecm',
        'CatBoost': 'forecast_value_catboost',
    }

    records = []
    grouped = merged_fc_df[merged_fc_df['horizon_days'] == horizon_days].groupby('forecast_target_date')

    for target_date, group in grouped:
        actual_val = _interp_actual_yield(df_filtered, target_date, maturity_years)
        if pd.isna(actual_val):
            continue

        for model_name, col in model_cols.items():
            if col not in group.columns:
                continue
            subset = group[['factor', col]].dropna()
            factors_dict = dict(zip(subset['factor'], subset[col]))
            if not {'beta0', 'beta1', 'beta2'}.issubset(factors_dict.keys()):
                continue
            if any(pd.isna([factors_dict['beta0'], factors_dict['beta1'], factors_dict['beta2']])):
                continue

            curve = NelsonSiegelCurve(
                beta0=factors_dict['beta0'],
                beta1=factors_dict['beta1'],
                beta2=factors_dict['beta2'],
                tau=tau1
            )
            y_pred = float(curve(maturity_years) * 100)
            records.append({
                'date': target_date,
                'model': model_name,
                'yield_forecast': y_pred,
                'actual_yield': actual_val,
            })

    df_plot = pd.DataFrame(records).sort_values('date')
    if df_plot.empty:
        print("Нет данных для построения графика доходностей")
        return

    if last_n_points is not None:
        unique_dates = df_plot['date'].drop_duplicates().sort_values()
        tail_dates = unique_dates.tail(last_n_points)
        df_plot = df_plot[df_plot['date'].isin(tail_dates)]

    plt.figure(figsize=(14, 6))
    actual_series = df_plot[['date', 'actual_yield']].drop_duplicates('date')
    plt.plot(actual_series['date'], actual_series['actual_yield'],
             color='black', linewidth=2.2, label='Факт')

    for model_name in df_plot['model'].unique():
        sub = df_plot[df_plot['model'] == model_name]
        plt.plot(sub['date'], sub['yield_forecast'], label=model_name, linewidth=1.4)

    plt.title(f"Доходность (maturity={maturity_years}y, horizon={horizon_days}д), tau1={tau1}")
    plt.xlabel('Дата целевой доходности')
    plt.ylabel('Доходность, %')
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()


# Пример вызова: 3 года, горизонт 21д, хвост 250 точек
plot_yield_forecasts_timeseries(
    merged_fc_df,
    df_filtered,
    tau1=best_tau1,
    maturity_years=3.0,
    horizon_days=21,
    last_n_points=250,
)
